# BDB27 — full-data game turns, counted on your laptop

**This notebook is the whole interface.** Put the folder holding your files in `DATA_DIR` (§1)
and press **Run All**. No terminal, no commands to type.

**What it produces.** The same turn table stage 02 builds, from your *full* game tracking files:
how many turns each position group actually makes in games, in figure 1's six bands — 30–60°,
60–90°, 90–120°, 120–150°, 150–180° and >180°. The detector stores 30–45° and 45–60° separately;
the 30–60° row is formed **only when reporting**, exactly as figure 1 does it.

**Your files are read as they are.** The 2027 release ships game tracking per season —
`game_tracking_2023.csv`, `game_tracking_2024.csv`, `game_tracking_2025.csv`. The notebook reads
every one that is present, without renaming or concatenating anything, and records each file's
season on every turn row it produces. A single concatenated `game_tracking.csv` is also accepted.

**Why it is trustworthy.** No detection maths lives in this notebook. It imports the pipeline's
own `Engine`, and reads every threshold from the repo's `config.yaml` (`turn_coverage:` block).
Before it touches full data it re-runs the pipeline on the repo's 1,048,575-row 2023 sample and
asserts the four published numbers plus a row-for-row match against
`outputs/figures/fig_the_hole.csv`. A mismatch **stops the notebook** with the exact diff.

**It never loads a tracking file into memory.** DuckDB converts each CSV to Parquet once, then
reads it back a group of games at a time (`resources.game_chunk_games`, 12 by default). Every
chunk is checkpointed, so closing the laptop and re-running resumes where it stopped.


## 1. Paths and mode — the only cell to edit

* `DATA_DIR` — the **folder** holding your `game_tracking_2023.csv`, `game_tracking_2024.csv`,
  `game_tracking_2025.csv`, `player_play.csv` and `players.csv`. `None` means the repo's own
  `data/raw`. Every season file that is present is read; nothing is renamed or concatenated.
  A single concatenated `game_tracking.csv` is accepted instead.
* If `combine_tracking.csv` and `combine_results.csv` sit in the same folder they are used too
  (they are cheap), so the Combine side of the 30–60° comparison is reported beside the game
  side. Their absence is never an error — the notebook says which side it could not compute.
* `INPUT_MODE` — `"full"` analyses your game tracking file(s); `"sample"` analyses the repo's 2023
  sample, which is what a dry run on this machine uses. The **fidelity gate always runs on the
  sample**, whatever this is set to.
* `INSTALL_MISSING` — leave it `False`. If §2 reports a package missing it prints the exact
  `pip` command and stops; set it to `True` only if you want the notebook to run that
  install for you. Nothing is ever installed without this flag.

Everything below §1 is fixed. Do not edit it.


In [ ]:
# ======================= CONFIG — the only cell to edit =========================
DATA_DIR   = None       # folder with game_tracking_2023/24/25.csv, player_play.csv, players.csv
INPUT_MODE = "full"     # "full" = your tracking file(s) | "sample" = repo 2023 sample
INSTALL_MISSING = False # True = let §2 install missing packages for you
# =============================================================================


## 2. Setup

Finds the repo (walks up until it sees `config.yaml` and `src/bdb27`), puts `src/` on the path,
and checks the libraries the pipeline needs. A package is installed **only** if it is genuinely
missing; an installed environment is left alone.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

_NEEDED = {"duckdb": "duckdb", "pandas": "pandas", "pyarrow": "pyarrow",
           "numpy": "numpy", "yaml": "pyyaml"}


def ensure_packages(missing, install):
    '''Report missing packages; install them ONLY when §1's INSTALL_MISSING is True.

    Nothing is ever installed silently, no shell is involved, and a failed install
    leaves the environment as it was and says what to run instead.
    '''
    if not missing:
        print("All required packages are already installed — nothing installed.")
        return
    names = ", ".join(missing)
    command = f'"{sys.executable}" -m pip install {names}'
    if not install:
        raise RuntimeError(
            f"Missing package(s): {names}. Nothing was installed.\n"
            f"  Install them yourself, then re-run this cell:\n      {command}\n"
            "  Or set INSTALL_MISSING = True in §1 and re-run.")
    print(f"INSTALL_MISSING=True — installing: {names} ...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    except (subprocess.CalledProcessError, OSError) as exc:
        raise RuntimeError(
            f"pip install failed for {names}: {exc}\n"
            f"  Install them yourself and re-run this cell:\n      {command}\n"
            "  (set INSTALL_MISSING back to False afterwards)") from exc
    print("install finished.")


ensure_packages([pip for mod, pip in _NEEDED.items()
                 if importlib.util.find_spec(mod) is None], INSTALL_MISSING)


def find_repo_root() -> Path:
    '''Walk upward from the notebook's working directory.'''
    here = Path.cwd().resolve()
    for parent in (here, *here.parents):
        if (parent / "config.yaml").exists() and (parent / "src" / "bdb27").is_dir():
            return parent
    raise RuntimeError(
        "Cannot locate the BDB27 repo root (need config.yaml + src/bdb27). "
        "Open this notebook from inside the BDB27 checkout.")


REPO = find_repo_root()
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from bdb27 import common
from bdb27.turns import TURN_COLUMNS, TURN_SCHEMA, Engine, frame_of

cfg = common.load_config(REPO / "config.yaml")
print(f"repo         : {REPO}")
print(f"config_sha   : {common.config_fingerprint(cfg)}")
print(f"git rev      : {common.git_rev(REPO)}")
print(f"chunk size   : {int(cfg.get('resources.game_chunk_games'))} games")

## 3. Inputs

Resolves the data folder and the tracking files it will read: one Parquet per season file
(or the single concatenated file). Prints a present/absent line for every input, the season
read from each tracking filename, and a SHA-256 per file (cached, so a resume does not re-read
gigabytes). A missing required file is always named — nothing is silently substituted, and a
folder that does not match the expected names is told exactly which names were looked for.


In [ ]:
import hashlib
import json
import re
import time

t_start = time.time()

#: Where the notebook writes everything. Already gitignored: nothing here is repo content.
RUN = REPO / "bdb27_run"
RUN.mkdir(exist_ok=True)
(RUN / "tmp").mkdir(exist_ok=True)

if INPUT_MODE not in ("full", "sample"):
    raise RuntimeError(f"INPUT_MODE must be 'full' or 'sample', got {INPUT_MODE!r}")

if DATA_DIR is None:
    DATA = REPO / "data" / "raw"
else:
    DATA = Path(DATA_DIR).expanduser().resolve()
    if not DATA.is_dir():
        raise RuntimeError(f"DATA_DIR does not exist or is not a folder: {DATA}")

#: Inputs the notebook reads from DATA. The full collection ships as one file per
#: season (data/README.md); a single concatenated file is accepted as well.
PP_CSV = DATA / cfg.get("files.player_play")
PLAYERS_CSV = DATA / cfg.get("files.players")
GAMES_CSV = DATA / cfg.get("files.games")
COMBINE_CSV = DATA / cfg.get("files.combine_tracking")
COMBINE_RESULTS_CSV = DATA / cfg.get("files.combine_results")
#: The repo's sample, used by the fidelity gate in §6 whatever INPUT_MODE says.
SAMPLE_CSV = REPO / "data" / "raw" / cfg.get("sample_files.game_tracking")
FIG_HOLE_CSV = REPO / "outputs" / "figures" / "fig_the_hole.csv"

TRACKING_COLUMNS = ["game_id", "play_id", "nfl_id", "time", "x", "y", "o", "dir"]


def season_of_tracking(path: Path):
    """The season encoded in a tracking filename, or None.

    `game_tracking_2023.csv` -> 2023; `game_tracking_2023_sample.csv` -> 2023;
    a concatenated `game_tracking.csv` carries no season in its name.
    """
    match = re.search(r"_(\d{4})(?:_sample)?$", Path(path).stem)
    return int(match.group(1)) if match else None


def tracking_inputs(data: Path, mode: str) -> list:
    """Every tracking CSV this mode reads, in season order — never the sample.

    The 2027 release ships game tracking as THREE per-season files. They are
    read exactly as named: no renaming and no concatenation. A single
    concatenated file named by `files.game_tracking` is accepted as an
    alternative. When none of the expected names exists, the error names every
    file that was looked for and every `game_tracking*.csv` that was found.
    """
    if mode == "sample":
        return [Path(data) / cfg.get("sample_files.game_tracking")]
    single = Path(data) / cfg.get("files.game_tracking")
    seasons = [Path(data) / name for name in cfg.get("files.game_tracking_seasons")]
    present = [p for p in seasons if p.exists()]
    if present:
        absent = [p.name for p in seasons if not p.exists()]
        if absent:
            print(f"  WARNING      season file(s) not found, so not analysed: {', '.join(absent)}")
        if single.exists():
            print(f"  note         {single.name} also present; the per-season files are used")
        return present
    if single.exists():
        return [single]
    found = sorted(p.name for p in Path(data).glob("game_tracking*.csv"))
    lines = [
        f"no game tracking file found in {data}",
        "  looked for, in this order:",
        *[f"    - {p.name}    (config.yaml files.game_tracking_seasons)" for p in seasons],
        f"    - {single.name}    (single concatenated file; config.yaml files.game_tracking)",
    ]
    if found:
        lines += ["  game_tracking*.csv files that ARE in the folder:"]
        lines += [f"    - {name}" for name in found]
        lines += ["  (none of these has an expected name — nothing was renamed, "
                  "concatenated or read)"]
    else:
        lines += ["  the folder holds no game_tracking*.csv at all"]
    lines += ["  will use     : nothing — no expected tracking file exists",
              "  the repo sample is NOT substituted here.",
              "  Put the full file(s) in this folder, or set INPUT_MODE = \"sample\"."]
    raise RuntimeError("\n".join(lines))


def sha256(path: Path) -> str:
    """SHA-256 of a file, read in 1 MiB binary blocks (no text encoding involved)."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def cached_sha256(path: Path) -> str:
    """`sha256(path)`, cached by (path, size, mtime) so a resume does not re-read GBs."""
    stat = Path(path).stat()
    key = f"{Path(path).resolve()}|{stat.st_size}|{stat.st_mtime_ns}"
    cache_path = RUN / "input_sha256.json"
    cache = (json.loads(cache_path.read_text(encoding="utf-8"))
             if cache_path.exists() else {})
    if key not in cache:
        print(f"  checksum {Path(path).name} ...", end=" ", flush=True)
        started = time.time()
        cache[key] = sha256(path)
        cache_path.write_text(json.dumps(cache, indent=2, sort_keys=True), encoding="utf-8")
        print(f"{time.time() - started:,.1f}s")
    return cache[key]


def count_rows(path: Path):
    """Rows in one CSV, counted by DuckDB; None when it cannot be read.

    Used only by the inventory print: a read problem is reported there and then
    left for the conversion step to raise properly.
    """
    try:
        con = common.connect(cfg)
        try:
            rows = con.execute(
                f"SELECT count(*) FROM read_csv_auto('{Path(path).as_posix()}', header=true, "
                f"sample_size={int(cfg.get('resources.csv_sniff_rows'))}, nullstr='NA')"
            ).fetchone()[0]
        finally:
            con.close()
        return int(rows)
    except Exception:
        return None


def tracking_inventory(data: Path):
    """Every expected tracking name, whether it is there, its size and row count.

    Printed BEFORE resolution, so a folder that does not match the expected names
    produces a readable list of what was looked for and what was found — never
    only a traceback. Files that are present but not expected are listed too and
    are never read.
    """
    single = Path(data) / cfg.get("files.game_tracking")
    seasons = [Path(data) / name for name in cfg.get("files.game_tracking_seasons")]
    entries = [("single concatenated", single)] + [("per-season", p) for p in seasons]
    inventory = []
    for kind, path in entries:
        exists = path.exists()
        inventory.append({
            "kind": kind, "path": path, "exists": exists,
            "bytes": path.stat().st_size if exists else None,
            "rows": count_rows(path) if exists else None,
        })
    expected = {entry["path"].name for entry in inventory}
    others = sorted(p.name for p in Path(data).glob("game_tracking*.csv")
                    if p.name not in expected)
    return inventory, others


def source_key(paths) -> str:
    """Identity of the set of tracking files, so a changed set cannot reuse chunks."""
    payload = "|".join(
        f"{Path(p).name}:{Path(p).stat().st_size}:{Path(p).stat().st_mtime_ns}"
        for p in paths)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


if INPUT_MODE == "full":
    INVENTORY, UNEXPECTED = tracking_inventory(DATA)
else:
    _path = DATA / cfg.get("sample_files.game_tracking")
    _exists = _path.exists()
    INVENTORY = [{"kind": "sample", "path": _path, "exists": _exists,
                  "bytes": _path.stat().st_size if _exists else None,
                  "rows": count_rows(_path) if _exists else None}]
    UNEXPECTED = []

for _role, _path in (("player_play (required)", PP_CSV),
                     ("players (required)", PLAYERS_CSV),
                     ("combine_tracking (optional)", COMBINE_CSV),
                     ("combine_results (optional)", COMBINE_RESULTS_CSV)):
    _exists = _path.exists()
    INVENTORY.append({"kind": _role, "path": _path, "exists": _exists,
                      "bytes": _path.stat().st_size if _exists else None,
                      "rows": count_rows(_path) if _exists else None})

print(f"\ninput inventory — {DATA}")
for _entry in INVENTORY:
    _state = "found " if _entry["exists"] else "ABSENT"
    _size = f"{_entry['bytes'] / 1e6:,.1f} MB" if _entry["exists"] else "-"
    _rows = (f"{_entry['rows']:,} rows" if _entry["rows"] is not None
             else ("unreadable/unknown" if _entry["exists"] else "-"))
    print(f"  {_state}  {_entry['path'].name:<34} {_entry['kind']:<26} "
          f"{_size:>10}  {_rows}")
if UNEXPECTED:
    print("  other game_tracking*.csv present (NOT expected, will NOT be read): "
          + ", ".join(UNEXPECTED))

TRACKING_CSVS = tracking_inputs(DATA, INPUT_MODE)
print(f"will use     : {', '.join(p.name for p in TRACKING_CSVS)}")

#: True when the single-file layout is used and its season has to come from
#: games.csv (its filename carries none). Keeps the checksum list and the
#: manifest in step with what is actually read.
GAMES_USED = (INPUT_MODE == "full" and len(TRACKING_CSVS) == 1
              and season_of_tracking(TRACKING_CSVS[0]) is None and GAMES_CSV.exists())
#: Per-season files that were expected but absent. Recorded in the manifest, so a
#: partial answer can never look like a complete one after the terminal scrolls away.
ABSENT_TRACKING = ([name for name in cfg.get("files.game_tracking_seasons")
                    if not (DATA / name).exists()] if INPUT_MODE == "full" else [])

missing = [p.name for p in (PP_CSV, PLAYERS_CSV) if not p.exists()]
missing += [p.name for p in TRACKING_CSVS if not p.exists()]
if missing:
    raise RuntimeError(f"Missing required input(s) in {DATA}: {', '.join(missing)}")

#: SHA-256 of every input file the run reads. The manifest names each one.
_optional = ([COMBINE_CSV, COMBINE_RESULTS_CSV]
             if COMBINE_CSV.exists() and COMBINE_RESULTS_CSV.exists() else [])
if GAMES_USED:
    _optional.append(GAMES_CSV)
INPUT_SHA = {p.name: cached_sha256(p)
             for p in [*TRACKING_CSVS, PP_CSV, PLAYERS_CSV, *_optional]}


## 4. Convert once — CSV → Parquet with DuckDB

Each tracking CSV is read once and written as its own Parquet, **sorted by
`game_id, play_id, nfl_id, time`** and carrying only the columns the detector uses
(`game_id, play_id, nfl_id, time, x, y, o, dir`) plus `season`. The season comes from the
filename (`game_tracking_2023.csv` → 2023); a single concatenated `game_tracking.csv` has no
season in its name, so its season is joined from `games.csv` when that file is present. Sorting
once is what makes every later read a narrow per-game read instead of a scan of the whole file.

Each step is checkpointed: if a Parquet exists, is newer than its CSV, and carries the columns
this notebook now needs (including `season`), conversion is skipped.


In [ ]:
import duckdb
import pyarrow.parquet as pq


def convert(csv_path: Path, columns, out_path: Path, order_by: str, *,
            season: int | None = None, join_games: Path | None = None,
            with_season: bool = False) -> Path:
    """One CSV -> one Parquet, checkpointed. Returns the Parquet path.

    `season` writes one constant SMALLINT season column. `join_games` instead
    takes the season per game from `games.csv`. `with_season` on its own writes
    a NULL season column, for a tracking file whose filename carries no season
    and for which no games.csv is available. Non-tracking inputs pass none of
    these and keep their schema unchanged.
    """
    wants_season = with_season or season is not None or join_games is not None
    reuse = out_path.exists() and out_path.stat().st_mtime >= csv_path.stat().st_mtime
    if reuse and join_games is not None:
        reuse = out_path.stat().st_mtime >= Path(join_games).stat().st_mtime
    if reuse and wants_season and "season" not in pq.read_schema(out_path).names:
        reuse = False
    if reuse:
        print(f"  reuse   {out_path.name}")
        return out_path
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = out_path.with_suffix(".parquet.tmp")
    con = common.connect(cfg)                       # memory limit + threads from config
    con.execute(f"SET temp_directory='{(RUN / 'tmp').as_posix()}'")
    con.execute("SET preserve_insertion_order=false")
    sniff = int(cfg.get("resources.csv_sniff_rows"))
    reader = (f"read_csv_auto('{csv_path.as_posix()}', header=true, "
              f"sample_size={sniff}, nullstr='NA')")
    if season is not None:
        select_sql = f"{', '.join(columns)}, CAST({int(season)} AS SMALLINT) AS season"
        from_sql, order_sql = reader, order_by
    elif join_games is not None:
        games_reader = (f"read_csv_auto('{Path(join_games).as_posix()}', header=true, "
                        f"sample_size={sniff}, nullstr='NA')")
        select_sql = (f"{', '.join('t.' + c for c in columns)}, "
                      "CAST(g.season AS SMALLINT) AS season")
        from_sql = f"{reader} AS t LEFT JOIN {games_reader} AS g ON t.game_id = g.game_id"
        order_sql = ", ".join("t." + c.strip() for c in order_by.split(","))
    elif with_season:
        select_sql = f"{', '.join(columns)}, CAST(NULL AS SMALLINT) AS season"
        from_sql, order_sql = reader, order_by
    else:
        select_sql, from_sql, order_sql = ", ".join(columns), reader, order_by
    t0 = time.time()
    con.execute(f'''
        COPY (
            SELECT {select_sql}
            FROM {from_sql}
            ORDER BY {order_sql}
        ) TO '{tmp.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
        ''')
    con.close()
    tmp.replace(out_path)
    print(f"  convert {out_path.name}  {time.time() - t0:,.1f}s")
    return out_path


_undeclared = [c for c in TRACKING_COLUMNS if c not in cfg.get("columns.game_tracking")]
if _undeclared:
    raise RuntimeError(f"config.yaml columns.game_tracking is missing {_undeclared}")

t_convert = time.time()
GAME_PQS = []
for _csv in TRACKING_CSVS:
    _season = season_of_tracking(_csv)
    _join = GAMES_CSV if GAMES_USED else None
    _how = (f"season {_season}" if _season is not None else
            ("season from games.csv" if _join is not None else "season unknown"))
    print(f"converting {_csv.name} ({_how}) ...")
    GAME_PQS.append(convert(_csv, TRACKING_COLUMNS, RUN / f"{_csv.stem}.parquet",
                            "game_id, play_id, nfl_id, time",
                            season=_season, join_games=_join, with_season=True))
PP_PQ = convert(PP_CSV, cfg.get("columns.player_play"), RUN / "player_play.parquet",
                "game_id, play_id, nfl_id")
PLAYERS_PQ = convert(PLAYERS_CSV, ["nfl_id", "display_name", "draft_year", "nfl_position"],
                     RUN / "players.parquet", "nfl_id")
COMBINE_PQ = (convert(COMBINE_CSV, cfg.get("columns.combine_tracking"),
                      RUN / "combine_tracking.parquet",
                      "drill_name, event_id, nfl_id, attempt, time")
              if COMBINE_CSV.exists() else None)
COMBINE_RESULTS_PQ = (convert(COMBINE_RESULTS_CSV, cfg.get("columns.combine_results"),
                              RUN / "combine_results.parquet", "nfl_id")
                      if COMBINE_CSV.exists() and COMBINE_RESULTS_CSV.exists() else None)
print(f"conversion total {time.time() - t_convert:,.1f}s")

paths_sql = "[" + ", ".join(f"'{p.as_posix()}'" for p in GAME_PQS) + "]"
_con = common.connect(cfg)
GAMES_ROWS = (int(_con.execute(
    f"SELECT count(*) FROM read_csv_auto('{GAMES_CSV.as_posix()}', header=true, nullstr='NA')"
).fetchone()[0]) if GAMES_USED else None)
n_rows, n_games, n_plays, n_players = _con.execute(f'''
    SELECT count(*), count(DISTINCT game_id),
           count(DISTINCT (game_id, play_id)), count(DISTINCT nfl_id)
    FROM read_parquet({paths_sql})''').fetchone()
TRACKING_STATS = []
for _csv, _pq in zip(TRACKING_CSVS, GAME_PQS):
    _rows, _games = _con.execute(
        f"SELECT count(*), count(DISTINCT game_id) FROM read_parquet('{_pq.as_posix()}')"
    ).fetchone()
    _seasons = [int(v) for v in _con.execute(
        f"SELECT DISTINCT season FROM read_parquet('{_pq.as_posix()}') WHERE season IS NOT NULL"
    ).fetchnumpy()["season"]]
    TRACKING_STATS.append({
        "name": _csv.name, "parquet": _pq.name, "rows": int(_rows), "games": int(_games),
        "seasons": sorted(_seasons),
        "season": int(_seasons[0]) if len(_seasons) == 1 else None,
        "bytes": _csv.stat().st_size, "sha256": INPUT_SHA[_csv.name],
    })
_con.close()
print(f"\ntracking parquet : {n_rows:,} rows | {n_games:,} games | "
      f"{n_plays:,} plays | {n_players:,} players")
for _stat in TRACKING_STATS:
    print(f"  {_stat['name']:<28} {_stat['rows']:>12,} rows  {_stat['games']:>5,} games  "
          f"season {','.join(map(str, _stat['seasons'])) or 'unknown'}")


## 5. Detect — the pipeline's own engine

`Engine.from_config(cfg, "game")` is imported from `src/bdb27/turns.py`: the smoothing, heading,
angular rate and turn segmentation are the ones stage 02 uses, with every threshold read from
`config.yaml`. Nothing is re-implemented here. This cell only builds the same context dictionary
stage 02 builds (`position_group` from `positions.game_families`, `alignment`/`snap_type` from
`player_play`, name and draft year from `players`) and feeds frames to the engine in game chunks.

Every turn row also carries `season`, taken from the tracking file the game came from, so a
per-season breakdown needs no second pass. If the tracking files disagree about a game's
season, the notebook refuses rather than mixing them.


In [ ]:
import json

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

ENGINE = Engine.from_config(cfg, "game")
POSITION_FAMILIES = cfg.get("positions.game_families")

#: The stage-02 turn schema plus `season`. The extra column is the ONLY change:
#: it makes a per-season breakdown possible later without re-running detection.
#: The Combine side carries season = null (the Combine is not a season).
NOTEBOOK_TURN_SCHEMA = TURN_SCHEMA.append(pa.field("season", pa.int16()))
NOTEBOOK_TURN_COLUMNS = [f.name for f in NOTEBOOK_TURN_SCHEMA]


def position_family(nfl_position) -> str:
    """Map an NFL position to its reporting family (config, single source of truth)."""
    for group, members in POSITION_FAMILIES.items():
        if nfl_position in members:
            return group
    return "OTHER"


def snap_type_sql(col: str = "") -> str:
    """The snap-type CASE from config's snap_types.rules (the rule set stage 02 uses)."""
    parts = []
    for rule in cfg.get("snap_types.rules"):
        test, name, column = rule["test"], rule["name"], rule.get("column")
        if test == "notnull":
            cond = f"{col}{column} IS NOT NULL"
        elif test == "truthy":
            cond = f"{col}{column}"
        elif test == "notnull_excluding":
            vals = ", ".join("'" + str(v) + "'" for v in rule["exclude"])
            cond = f"({col}{column} IS NOT NULL AND {col}{column} NOT IN ({vals}))"
        else:
            cond = "TRUE"
        parts.append(f"WHEN {cond} THEN '{name}'")
    return "CASE " + " ".join(parts) + " ELSE 'OTHER' END"


def load_players(con) -> dict:
    df = con.execute(
        f"SELECT nfl_id, display_name, draft_year, nfl_position FROM read_parquet('{PLAYERS_PQ.as_posix()}')"
    ).df()
    return df.set_index("nfl_id").to_dict("index")


def draft_year_of(players: dict, nfl_id: int):
    """Draft year, or None — pandas gives NaN for a missing year, which is not an int."""
    value = players.get(nfl_id, {}).get("draft_year")
    return int(value) if value is not None and pd.notna(value) else None


def arrow_table(frame: pd.DataFrame) -> pa.Table:
    """Cast a turn frame to the notebook's schema (stage 02's + `season`)."""
    arrays = [common._arrow_array(frame[f.name], f.type) for f in NOTEBOOK_TURN_SCHEMA]
    return pa.Table.from_arrays(arrays, schema=NOTEBOOK_TURN_SCHEMA)


def write_part(rows: list, part: Path) -> int:
    tmp = part.with_suffix(".parquet.tmp")
    frame = frame_of(rows, NOTEBOOK_TURN_SCHEMA)
    pq.write_table(arrow_table(frame), tmp, compression="zstd")
    tmp.replace(part)
    return len(frame)


def detect_game_chunks(outdir: Path, tracking_pqs: list, label: str) -> list:
    """Run the engine over every (game, play, player), one group of games at a time.

    `tracking_pqs` is every per-season Parquet, read together. Season is read
    from the tracking rows, put in the engine context and therefore written on
    every turn row. Writes one checkpoint Parquet per chunk plus a JSON receipt,
    so a re-run skips completed chunks; a chunk is reused only when the config
    fingerprint AND the set of tracking files (`source_key`) are unchanged.
    """
    tracking_pqs = [Path(p) for p in tracking_pqs]
    paths_sql = "[" + ", ".join(f"'{p.as_posix()}'" for p in tracking_pqs) + "]"
    outdir.mkdir(parents=True, exist_ok=True)
    con = common.connect(cfg)
    games = [int(g) for g in con.execute(
        f"SELECT DISTINCT game_id FROM read_parquet({paths_sql}) ORDER BY 1"
    ).fetchnumpy()["game_id"]]
    season_rows = con.execute(f'''
        SELECT game_id, count(DISTINCT season) AS n_seasons, min(season) AS season
        FROM read_parquet({paths_sql}) GROUP BY 1''').df()
    clashes = season_rows[season_rows["n_seasons"] > 1]
    if len(clashes):
        raise RuntimeError(
            "the tracking files disagree about the season of "
            f"{len(clashes)} game(s), e.g. game_id {int(clashes.iloc[0]['game_id'])} — "
            "refusing to mix them")
    season_of_game = {int(r.game_id): (None if pd.isna(r.season) else int(r.season))
                      for r in season_rows.itertuples()}
    size = int(cfg.get("resources.game_chunk_games"))
    players = load_players(con)
    sha = common.config_fingerprint(cfg)
    key = source_key(tracking_pqs)

    receipts, t0, done_rows = [], time.time(), 0
    n_chunks = (len(games) + size - 1) // size
    print(f"{label}: {len(games):,} games in {n_chunks} chunk(s) of {size}")
    for k, start in enumerate(range(0, len(games), size)):
        batch = games[start:start + size]
        part = outdir / f"chunk_{k:05d}.parquet"
        receipt_path = outdir / f"chunk_{k:05d}.json"
        if part.exists() and receipt_path.exists():
            old = json.loads(receipt_path.read_text(encoding="utf-8"))
            if (old.get("config_sha") == sha and old.get("n_games") == len(batch)
                    and old.get("source_key") == key):
                receipts.append(old)
                done_rows += int(old["rows"])
                print(f"  [{k + 1}/{n_chunks}] reused  chunk {k:05d} ({old['rows']:,} turns)")
                continue
        t_chunk = time.time()
        ph = ",".join("?" * len(batch))
        df = con.execute(f'''
            SELECT game_id, play_id, nfl_id, epoch_ms(time) AS ms, x, y, o, dir
            FROM read_parquet({paths_sql}) WHERE game_id IN ({ph})''', [*batch]).df()
        df = df.sort_values(["game_id", "play_id", "nfl_id", "ms"], kind="stable")
        pp = con.execute(f'''
            SELECT game_id, play_id, nfl_id, lined_up_position,
                   {snap_type_sql()} AS snap_type
            FROM read_parquet(?) WHERE game_id IN ({ph})''', [str(PP_PQ), *batch]).df()
        pp_idx = pp.set_index(["game_id", "play_id", "nfl_id"]).to_dict("index")
        rows, n_snaps = [], 0
        for (gid, pid, nid), g in df.groupby(["game_id", "play_id", "nfl_id"],
                                             observed=True, sort=True):
            info = pp_idx.get((gid, pid, nid), {})
            nid = int(nid)
            npos = players.get(nid, {}).get("nfl_position")
            ctx = {
                "drill_name": None, "event_id": None, "attempt": None,
                "game_id": int(gid), "play_id": int(pid), "nfl_id": nid,
                "player_name": players.get(nid, {}).get("display_name"),
                "position": npos, "position_group": position_family(npos),
                "alignment": info.get("lined_up_position"),
                "snap_type": info.get("snap_type"),
                "draft_year": draft_year_of(players, nid),
                "season": season_of_game.get(int(gid)),
            }
            _att, trs = ENGINE.analyse(g.ms.to_numpy(), g.x.to_numpy(), g.y.to_numpy(), ctx,
                                       o=g.o.to_numpy(), dirv=g.dir.to_numpy())
            rows.extend(trs)
            n_snaps += 1
        rows_written = write_part(rows, part)
        dt = time.time() - t_chunk
        receipt = {
            "chunk": k, "label": label, "games": [int(batch[0]), int(batch[-1])],
            "n_games": len(batch), "snaps": n_snaps, "rows": rows_written,
            "frames": int(len(df)), "seconds": round(dt, 2), "config_sha": sha,
            "source_key": key,
            "seasons": sorted({s for s in (season_of_game.get(int(g)) for g in batch)
                               if s is not None}),
        }
        receipt_path.write_text(json.dumps(receipt, indent=2), encoding="utf-8")
        receipts.append(receipt)
        done_rows += rows_written
        elapsed = time.time() - t0
        eta = elapsed / (k + 1) * (n_chunks - k - 1)
        print(f"  [{k + 1}/{n_chunks}] games {batch[0]}..{batch[-1]}  "
              f"{len(df):,} rows -> {rows_written:,} turns  {dt:,.1f}s  "
              f"| elapsed {elapsed / 60:,.1f} min | ETA {eta / 60:,.1f} min")
    con.close()
    print(f"{label}: {done_rows:,} turn rows in {n_chunks} chunk(s)")
    return receipts


### Combine side (optional — used only when both its files are present)

The same engine with `setting="combine"`, fed from `combine_tracking.csv`. It supplies the
Combine's own 30–60° share, which is the number the game share is read against. Combine turn
rows carry `season = null`: a Combine event is not a season.


In [ ]:
def detect_combine() -> list:
    '''Run the engine over the Combine attempts (one context row per attempt).'''
    if COMBINE_PQ is None or COMBINE_RESULTS_PQ is None:
        print("combine: skipped (combine_tracking.csv / combine_results.csv not present)")
        return []
    con = common.connect(cfg)
    engine = Engine.from_config(cfg, "combine")
    players = load_players(con)
    pos_map = con.execute(
        f"SELECT nfl_id, combine_position FROM read_parquet('{COMBINE_RESULTS_PQ.as_posix()}')"
    ).df().set_index("nfl_id").combine_position.to_dict()
    df = con.execute(f'''
        SELECT drill_name, event_id, nfl_id, attempt, epoch_ms(time) AS ms, x, y
        FROM read_parquet('{COMBINE_PQ.as_posix()}')
        WHERE entity_type = 'PLAYER'
        ORDER BY drill_name, event_id, nfl_id, attempt, ms''').df()
    dup = df.duplicated(["nfl_id", "event_id", "attempt", "ms"])
    dup_by_key = (df[dup].groupby(["event_id", "nfl_id", "attempt"]).size().to_dict()
                  if bool(dup.any()) else {})
    if cfg.get("sampling.drop_duplicate_frames"):
        df = df[~dup]
    rows, t0 = [], time.time()
    for (drill, event_id, nfl_id, attempt), g in df.groupby(
            ["drill_name", "event_id", "nfl_id", "attempt"], observed=True, sort=False):
        nid = int(nfl_id)
        ctx = {
            "drill_name": drill, "event_id": str(event_id), "attempt": int(attempt),
            "game_id": None, "play_id": None, "nfl_id": nid,
            "player_name": players.get(nid, {}).get("display_name"),
            "position": pos_map.get(nid), "position_group": pos_map.get(nid),
            "alignment": None, "snap_type": None,
            "draft_year": draft_year_of(players, nid),
            "season": None,          # a Combine event is not a season
        }
        _att, trs = engine.analyse(
            g.ms.to_numpy(), g.x.to_numpy(), g.y.to_numpy(), ctx,
            duplicate_frames=int(dup_by_key.get((event_id, nfl_id, attempt), 0)))
        rows.extend(trs)
    con.close()
    part = RUN / "combine_turns.parquet"
    n = write_part(rows, part)
    print(f"combine: {n:,} turn rows from {len(df):,} frames in {time.time() - t0:,.1f}s")
    return rows

## 6. Fidelity gate — must pass before any full-data work

The gate re-runs §4–§5 on the repo's 2023 sample (1,048,575 rows — the vendor's 2^20 − 1 row cap
— 333 games, 4,228 plays, 154 players) and requires **all** of:

* **2,262** valid DB game turns,
* **326** of them in the merged 30–60° band,
* **33** distinct DB players,
* **299** DB player-games,
* a row-for-row match against `outputs/figures/fig_the_hole.csv` — count, `setting_total` and
  `share_within_setting`, band for band, both settings when the Combine files are present.

Any mismatch prints the exact diff and **stops the notebook**: no full-data number is ever
produced past a failed gate. The gate has its own checkpoint folder, so its second run is seconds.

In [ ]:
EXPECTED = {
    "valid DB game turns": 2262,
    "DB game turns, 30-60 deg (merged)": 326,
    "DB game players": 33,
    "DB player-games": 299,
}
BAND_EDGES = [float(x) for x in cfg.get("figures.hole.bands.turn_angle_deg")]
BAND_OVERFLOW = cfg.get("figures.hole.bands.overflow_label")
BAND_LABELS = ([f"{lo:g}–{hi:g}°" for lo, hi in zip(BAND_EDGES[:-1], BAND_EDGES[1:])]
               + [f"{BAND_OVERFLOW}°"])
GATE_READY = False


def band_of(angle: pd.Series) -> pd.Series:
    '''Left-closed band labels from config; the last band is everything at/above the top edge.'''
    labels = pd.Series([None] * len(angle), index=angle.index, dtype=object)
    for (lo, hi), name in zip(zip(BAND_EDGES[:-1], BAND_EDGES[1:]), BAND_LABELS[:-1]):
        labels[(angle >= lo) & (angle < hi)] = name
    labels[angle >= BAND_EDGES[-1]] = BAND_LABELS[-1]
    return labels


def bands_table(turns: pd.DataFrame) -> pd.DataFrame:
    '''position_group x band x setting over VALID turns. Shares are within the setting.'''
    valid = turns[turns["valid"].fillna(False).astype(bool)].copy()
    valid["band"] = band_of(pd.to_numeric(valid["abs_turn_angle"], errors="coerce"))
    if valid["band"].isna().any():
        raise RuntimeError("some valid turns fall outside the configured angle bands")
    totals = (valid.groupby(["position_group", "setting"]).size()
              .rename("setting_total").reset_index())
    counts = (valid.groupby(["position_group", "band", "setting"]).size()
              .rename("count").reset_index())
    out = counts.merge(totals, on=["position_group", "setting"], how="left")
    out["band_index"] = out["band"].map({b: i + 1 for i, b in enumerate(BAND_LABELS)})
    out["band_low_deg"] = [BAND_EDGES[int(i) - 1] for i in out["band_index"]]
    out["band_high_deg"] = [BAND_EDGES[int(i)] if int(i) < len(BAND_EDGES) else None
                            for i in out["band_index"]]
    out["share_within_setting"] = out["count"] / out["setting_total"]
    cols = ["setting", "position_group", "band", "band_index", "band_low_deg",
            "band_high_deg", "count", "setting_total", "share_within_setting"]
    return (out[cols].sort_values(["setting", "position_group", "band_index"], kind="stable")
            .reset_index(drop=True))


REPORT_BANDS = ["30–60°"] + BAND_LABELS[2:]
REPORT_EDGES = [BAND_EDGES[0]] + BAND_EDGES[2:]


def merge_30_60(table: pd.DataFrame) -> pd.DataFrame:
    '''Reporting frame: 30–45° and 45–60° summed into 30–60°, every other band as stored.

    Called only when reporting — §6's comparison and §8's artefact. The stored table keeps
    30–45° and 45–60° separate; it is never merged.
    '''
    low = table[table["band"].isin(BAND_LABELS[:2])]
    merged = (low.groupby(["position_group", "setting"], as_index=False)
                 .agg(count=("count", "sum"), setting_total=("setting_total", "first")))
    merged["band"] = REPORT_BANDS[0]
    out = pd.concat([merged, table[table["band"].isin(BAND_LABELS[2:])]], ignore_index=True)
    out["share_within_setting"] = out["count"] / out["setting_total"]
    out["band_index"] = out["band"].map({b: i + 1 for i, b in enumerate(REPORT_BANDS)})
    out["band_low_deg"] = [REPORT_EDGES[int(i) - 1] for i in out["band_index"]]
    out["band_high_deg"] = [REPORT_EDGES[int(i)] if int(i) < len(REPORT_EDGES) else None
                            for i in out["band_index"]]
    cols = ["setting", "position_group", "band", "band_index", "band_low_deg",
            "band_high_deg", "count", "setting_total", "share_within_setting"]
    return (out[cols].sort_values(["setting", "position_group", "band_index"], kind="stable")
            .reset_index(drop=True))


gate_turns, table = None, None
if not SAMPLE_CSV.exists():
    print("GATE SKIPPED — the repo sample is not present on this machine.")
    print(f"  expected at : {SAMPLE_CSV}")
    print("  The gate passed on the authoring host (git rev dc595f2; notes/decisions.md D184).")
    print("  To run it here, copy game_tracking_2023_sample.csv to that path and re-run §6.")
else:
    print(f"gate sample: {SAMPLE_CSV}")
    gate_dir = RUN / "gate"
    SAMPLE_PQ = convert(SAMPLE_CSV, TRACKING_COLUMNS,
                        gate_dir / f"{SAMPLE_CSV.stem}.parquet",
                        "game_id, play_id, nfl_id, time",
                        season=season_of_tracking(SAMPLE_CSV))
    gate_receipts = detect_game_chunks(gate_dir / "parts", [SAMPLE_PQ], "gate-game")
    gate_parts = sorted((gate_dir / "parts").glob("chunk_*.parquet"))
    gate_turns = pd.concat([pd.read_parquet(p) for p in gate_parts], ignore_index=True)
    if gate_turns.empty:
        raise RuntimeError("gate produced no turn rows at all — refusing to continue")
    combine_rows = detect_combine()
    if combine_rows:
        gate_turns = pd.concat([gate_turns,
                               pd.DataFrame(combine_rows, columns=NOTEBOOK_TURN_COLUMNS)],
                               ignore_index=True)
    gate_turns["valid"] = gate_turns["valid"].fillna(False).astype(bool)

    db = gate_turns[(gate_turns["setting"] == "game")
                    & (gate_turns["position_group"] == "DB") & gate_turns["valid"]]
    got = {
        "valid DB game turns": int(len(db)),
        "DB game turns, 30-60 deg (merged)": int(((db["abs_turn_angle"] >= 30)
                                                  & (db["abs_turn_angle"] < 60)).sum()),
        "DB game players": int(db["nfl_id"].nunique()),
        "DB player-games": int(db.groupby(["nfl_id", "game_id"]).ngroups),
    }
    print("\ngate numbers")
    bad = []
    for key, want in EXPECTED.items():
        ok = got[key] == want
        bad += [] if ok else [key]
        print(f"  {'PASS' if ok else 'FAIL'}  {key:<38} expected {want:>6,}  got {got[key]:>6,}")

    ref = pd.read_csv(FIG_HOLE_CSV, encoding="utf-8")
    ref = ref[ref["figure"] == "fig_the_hole"]
    REF_ROWS = int(len(ref))
    table = bands_table(gate_turns)
    merged = merge_30_60(table)
    diffs, compared, skipped = [], set(), set()
    available = set(merged["setting"])
    combine_absent = COMBINE_PQ is None or COMBINE_RESULTS_PQ is None
    for _, row in ref.iterrows():
        if row["setting"] not in available:
            if row["setting"] == "combine" and combine_absent:
                skipped.add("combine")      # optional side absent, so not comparable
                continue
            diffs.append(f"{row['setting']}/{row['band']}: absent from the notebook's table")
            continue
        cand = merged[(merged["position_group"] == "DB")
                      & (merged["setting"] == row["setting"])
                      & (merged["band"] == row["band"])]
        if cand.empty:
            diffs.append(f"{row['setting']}/{row['band']}: absent from the notebook's table")
            continue
        compared.add(row["setting"])
        g = cand.iloc[0]
        for col in ("count", "setting_total"):
            if int(g[col]) != int(row[col]):
                diffs.append(f"{row['setting']}/{row['band']}/{col}: published "
                             f"{int(row[col]):,} vs notebook {int(g[col]):,}")
        if abs(float(g["share_within_setting"]) - float(row["share_within_setting"])) > 1e-9:
            diffs.append(f"{row['setting']}/{row['band']}/share: published "
                         f"{row['share_within_setting']:.6f} vs notebook "
                         f"{g['share_within_setting']:.6f}")
    print(f"\nrow-for-row vs {FIG_HOLE_CSV.name}: {len(ref)} published rows, "
          f"{len(diffs)} difference(s); settings compared: {sorted(compared)}")
    if skipped:
        print(f"  (settings not compared — their input files are absent: "
              f"{', '.join(sorted(skipped))})")
    for d in diffs:
        print(f"  DIFF  {d}")

    print("\nDB game bands — 30–45° and 45–60° kept separate (the stored table, unmerged)")
    db_bands = table[(table["setting"] == "game") & (table["position_group"] == "DB")]
    for _, r in db_bands.iterrows():
        print(f"  {r['band']:<12} {int(r['count']):>5,}  of {int(r['setting_total']):,}  "
              f"({100 * r['share_within_setting']:5.2f}%)")
    m = merged[(merged["position_group"] == "DB") & (merged["setting"] == "game")]
    if not m.empty:
        print(f"  merged 30–60°  {int(m.iloc[0]['count']):>5,}  of "
              f"{int(m.iloc[0]['setting_total']):,}  "
              f"({100 * m.iloc[0]['share_within_setting']:5.2f}%)")

    if bad or diffs:
        print("\nGATE FAILED — stopping before any full-data work.")
        print("  Nothing was written to the full-data folders. The diff above is what needs"
              " fixing first.")
        raise RuntimeError(f"fidelity gate failed: {bad + diffs}")
    print("\n" + "-" * 76)
    print("  FIDELITY GATE PASSED — this notebook reproduces the published sample exactly")
    print("-" * 76)
    GATE_READY = True

## 7. The full run — chunked, resumable, with an ETA

Runs the engine over every tracking file `INPUT_MODE` names. Each group of
`resources.game_chunk_games` games is written to its own checkpoint, so a closed laptop resumes
at the next unfinished chunk. Progress prints elapsed time and an ETA. No tracking file is ever
loaded whole.


In [ ]:
run_dir = RUN / f"run_{INPUT_MODE}"
print(f"run folder: {run_dir}\n")
t_run = time.time()
receipts = detect_game_chunks(run_dir / "parts", GAME_PQS, f"run-{INPUT_MODE}")
n_seconds = sum(float(r["seconds"]) for r in receipts)
n_frames = sum(int(r["frames"]) for r in receipts)
print(f"run complete: {time.time() - t_run:,.1f}s wall, "
      f"{n_seconds / 60:,.1f} min of chunk time, {n_frames:,} frames")
if n_frames:
    print(f"throughput: {n_frames / max(n_seconds, 1e-9):,.0f} frames/s")


## 8. Artefacts

Reads every checkpoint back (they are small), applies the reporting bands, and writes three
files into `bdb27_run/artefacts/`. Nothing outside `bdb27_run/` is touched: no existing figure,
CSV or manifest in `outputs/` changes. The DB turn rows carry the pipeline's turn columns plus
`season`; the manifest names every input file it read with its row count and SHA-256, and
reports turn counts by season.


In [ ]:
ART = RUN / "artefacts"
ART.mkdir(parents=True, exist_ok=True)

parts = sorted((run_dir / "parts").glob("chunk_*.parquet"))
if not parts:
    raise RuntimeError("no chunk checkpoints to assemble")
turns = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
if INPUT_MODE == "full":
    combine_rows = detect_combine()
    if combine_rows:
        turns = pd.concat([turns, pd.DataFrame(combine_rows, columns=NOTEBOOK_TURN_COLUMNS)],
                          ignore_index=True)
turns["valid"] = turns["valid"].fillna(False).astype(bool)
if turns.empty:
    raise RuntimeError("the run produced no turn rows — nothing to write")
if "season" not in turns.columns:
    turns["season"] = pd.NA

# ---- 1. bands: position_group x band x setting, figure 1's six reporting bands ----------
bands = merge_30_60(bands_table(turns))
bands_out = ART / "bdb27_full_game_bands.csv"
bands.to_csv(bands_out, index=False, encoding="utf-8")
print(f"bands    : {bands_out.name}  {bands_out.stat().st_size:,} bytes  ({len(bands)} rows)")

# ---- 2. DB turn rows, the pipeline's turn schema plus `season` -------------------------
db_turns = turns[(turns["setting"] == "game")
                 & (turns["position_group"] == "DB")][NOTEBOOK_TURN_COLUMNS].reset_index(drop=True)
turns_out = ART / "bdb27_full_db_turns.parquet"
pq.write_table(arrow_table(db_turns), turns_out, compression="zstd")
print(f"DB turns : {turns_out.name}  {turns_out.stat().st_size:,} bytes  "
      f"({len(db_turns):,} rows, season column included)")


# ---- 3. manifest ------------------------------------------------------------------------
game = turns[turns["setting"] == "game"]
db_valid = game[(game["position_group"] == "DB") & game["valid"]]


def by_season(frame: pd.DataFrame) -> dict:
    """Row counts per season; a missing season is counted as "unknown"."""
    if frame.empty:
        return {}
    labels = frame["season"].map(lambda v: "unknown" if pd.isna(v) else str(int(v)))
    counts = labels.value_counts()
    return {str(k): int(counts[k]) for k in sorted(counts.index,
                                                   key=lambda k: (k == "unknown", k))}


_con = common.connect(cfg)


def parquet_rows(path: Path) -> int:
    return int(_con.execute(
        f"SELECT count(*) FROM read_parquet('{Path(path).as_posix()}')").fetchone()[0])


input_files = [dict(stat, role="game_tracking") for stat in TRACKING_STATS]
input_files.append({"name": PP_CSV.name, "role": "player_play",
                    "bytes": PP_CSV.stat().st_size, "sha256": INPUT_SHA[PP_CSV.name],
                    "rows": parquet_rows(PP_PQ)})
input_files.append({"name": PLAYERS_CSV.name, "role": "players",
                    "bytes": PLAYERS_CSV.stat().st_size, "sha256": INPUT_SHA[PLAYERS_CSV.name],
                    "rows": parquet_rows(PLAYERS_PQ)})
if COMBINE_PQ is not None:
    input_files.append({"name": COMBINE_CSV.name, "role": "combine_tracking",
                        "bytes": COMBINE_CSV.stat().st_size,
                        "sha256": INPUT_SHA[COMBINE_CSV.name],
                        "rows": parquet_rows(COMBINE_PQ)})
if COMBINE_RESULTS_PQ is not None:
    input_files.append({"name": COMBINE_RESULTS_CSV.name, "role": "combine_results",
                        "bytes": COMBINE_RESULTS_CSV.stat().st_size,
                        "sha256": INPUT_SHA[COMBINE_RESULTS_CSV.name],
                        "rows": parquet_rows(COMBINE_RESULTS_PQ)})
if GAMES_USED:
    input_files.append({"name": GAMES_CSV.name, "role": "games",
                        "bytes": GAMES_CSV.stat().st_size,
                        "sha256": INPUT_SHA[GAMES_CSV.name], "rows": int(GAMES_ROWS)})
_con.close()

gate_parquet = RUN / "gate" / f"{SAMPLE_CSV.stem}.parquet"
gate_record = {
    "name": SAMPLE_CSV.name,
    "role": "fidelity_gate_sample",
    "path": str(SAMPLE_CSV),
    "present": SAMPLE_CSV.exists(),
    "bytes": SAMPLE_CSV.stat().st_size if SAMPLE_CSV.exists() else None,
    "sha256": cached_sha256(SAMPLE_CSV) if SAMPLE_CSV.exists() else None,
    "rows": int(pq.read_metadata(gate_parquet).num_rows) if gate_parquet.exists() else None,
}

manifest = {
    "notebook": "bdb27_full_game_turns.ipynb",
    "input_mode": INPUT_MODE,
    "input_file": ", ".join(p.name for p in TRACKING_CSVS),
    "input_files": input_files,
    "season_files_absent": ABSENT_TRACKING,
    "gate_sample": gate_record,
    "gate_reference": {
        "name": FIG_HOLE_CSV.name,
        "path": str(FIG_HOLE_CSV),
        "bytes": FIG_HOLE_CSV.stat().st_size if FIG_HOLE_CSV.exists() else None,
        "sha256": cached_sha256(FIG_HOLE_CSV) if FIG_HOLE_CSV.exists() else None,
        "rows_compared": globals().get("REF_ROWS"),
    },
    "season_source": ("filename game_tracking_<season>.csv; games.csv when a single "
                      "game_tracking.csv has no season in its name"),
    "config_sha": common.config_fingerprint(cfg),
    "git_rev": common.git_rev(REPO),
    "primary_smoother": ENGINE.primary,
    "speed_floor_game": float(cfg.get("detector.speed_floor_game")),
    "chunk_games": int(cfg.get("resources.game_chunk_games")),
    "fidelity_gate": "passed" if GATE_READY else "skipped-absent",
    "counts": {
        "tracking_rows": int(n_rows), "games": int(game["game_id"].nunique()),
        "plays": int(game.groupby(["game_id", "play_id"]).ngroups),
        "players": int(game["nfl_id"].nunique()),
        "player_games": int(game.groupby(["nfl_id", "game_id"]).ngroups),
        "turn_rows": int(len(turns)),
        "valid_db_turns": int(len(db_valid)),
        "valid_db_players": int(db_valid["nfl_id"].nunique()),
        "valid_db_player_games": int(db_valid.groupby(["nfl_id", "game_id"]).ngroups),
        "valid_db_30_60": int(((db_valid["abs_turn_angle"] >= 30)
                               & (db_valid["abs_turn_angle"] < 60)).sum()),
        "by_season": {
            "game_turns": by_season(game),
            "valid_db_turns": by_season(db_valid),
        },
    },
    "chunks": receipts,
    "artefacts": {
        bands_out.name: {"bytes": bands_out.stat().st_size, "sha256": sha256(bands_out)},
        turns_out.name: {"bytes": turns_out.stat().st_size, "sha256": sha256(turns_out)},
    },
    "runtime_seconds": round(time.time() - t_start, 1),
}
man_out = ART / "bdb27_full_game_manifest.json"
man_out.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"manifest : {man_out.name}  {man_out.stat().st_size:,} bytes")

print(f"\nfolder to upload: {ART}")
print(json.dumps(manifest["counts"], indent=2))


## 9. What to send back

Send these three files from the folder printed above (`BDB27/bdb27_run/artefacts/`):

1. `bdb27_full_game_bands.csv` — the counts by position group, band and setting.
2. `bdb27_full_db_turns.parquet` — the DB turn rows, one row per turn: the same columns as
   `outputs/turns/turns.parquet` plus `season` (the tracking file the turn came from; null on
   the Combine side).
3. `bdb27_full_game_manifest.json` — modes, config fingerprint, git revision, every count, the
   per-season counts, every input file with its row count and SHA-256, the per-chunk timings and
   the SHA-256 of the other two files.

If §6 printed a red `GATE FAILED` line, **send nothing**: the notebook stopped on purpose and the
diff above is what needs fixing first.
